# Multimodal Circuit Breaker — Qwen2-VL-2B (Colab T4)

**Runtime → Change runtime type → GPU (T4 or better).**

Upload `data/colab_upload.zip` (built on the Mac by `src/pack_for_colab.py`).

Adapters for this run live under **`out/qwen2vl-cb-v2/`** (not `out/qwen2vl-cb/`).

In [ ]:
# 1. Environment
!nvidia-smi
!pip -q install -U transformers accelerate peft qwen-vl-utils pillow scikit-learn joblib matplotlib

In [ ]:
# 2. Upload colab_upload.zip (images + manifests + src/)
from google.colab import files
uploaded = files.upload()
!unzip -q -o colab_upload.zip
!ls src data images/train | head

In [ ]:
# 3. Train the circuit breaker (~2-4 h on T4 for run v2)
# Filter cache is reused if present. batch-size 1 fits T4 16GB.
# Watch cb_cosine — should fall well below 0.9 with RMU.
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
!python src/train_cb_vlm.py \
    --manifest data/manifest_train.json \
    --image-root . \
    --output-dir out/qwen2vl-cb-v2 \
    --alpha 10 --cb-coeff-floor 0.2 \
    --max-steps 500 --batch-size 1 --lr 2e-4 \
    --cb-loss rmu --lora-b-init 1e-3 \
    --filter-complying \
    --complying-cache out/qwen2vl-cb-v2/complying_cache.json \
    --log-every 10 --save-every 100

In [ ]:
# 4. Sanity generations: harmful should short-circuit, benign should answer
import json, torch
from pathlib import Path
from PIL import Image
from peft import PeftModel
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

ADAPTER = 'out/qwen2vl-cb-v2/adapter'
if not Path(ADAPTER, 'adapter_config.json').exists():
    ckpts = sorted(Path('out/qwen2vl-cb-v2').glob('step*/adapter_config.json'))
    if not ckpts:
        raise FileNotFoundError(
            'No adapter under out/qwen2vl-cb-v2/. '
            'Training may still be running, or path is wrong. '
            f'Contents: {list(Path("out").rglob("adapter_config.json"))}'
        )
    ADAPTER = str(ckpts[-1].parent)
print('loading adapter from', ADAPTER)

base = Qwen2VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2-VL-2B-Instruct', torch_dtype=torch.float16, device_map='cuda')
model = PeftModel.from_pretrained(base, ADAPTER)
proc_src = ADAPTER if Path(ADAPTER, 'tokenizer_config.json').exists() else 'Qwen/Qwen2-VL-2B-Instruct'
processor = AutoProcessor.from_pretrained(proc_src)

manifest = json.load(open('data/manifest_eval.json'))

def gen(item, use_adapter=True):
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': item['image']},
        {'type': 'text', 'text': item['carrier']}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[Image.open(item['image']).convert('RGB')],
                       return_tensors='pt').to('cuda')
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
        else:
            with model.disable_adapter():
                out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]

harm = next(x for x in manifest if x['label'] == 'harmful')
ben = next(x for x in manifest if x['label'] == 'benign')
print('=== HARMFUL, BASE ==='); print(gen(harm, False)[:500])
print('\n=== HARMFUL, CIRCUIT-BROKEN ==='); print(gen(harm, True)[:500])
print('\n=== BENIGN, CIRCUIT-BROKEN ==='); print(gen(ben, True)[:500])
del base, model
torch.cuda.empty_cache()

In [ ]:
# 5. Before/after eval on held-out images (clean + perturbations) — GPU
import sys
from pathlib import Path
sys.path.insert(0, 'src')
ADAPTER = 'out/qwen2vl-cb-v2/adapter'
if not Path(ADAPTER, 'adapter_config.json').exists():
    ADAPTER = str(sorted(Path('out/qwen2vl-cb-v2').glob('step*'))[-1])
!python src/eval_cb.py --adapter {ADAPTER} --limit 20

In [ ]:
# 6. Activation probe (secondary) — GPU forwards, CPU sklearn
!python src/extract_activations.py --split train
!python src/extract_activations.py --split eval
!python src/train_probe.py
!python src/perturb_eval.py --limit 30

In [ ]:
# 7. Save everything to Drive + download
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/qwen2vl-cb-v2
!cp -r out/qwen2vl-cb-v2 /content/drive/MyDrive/
!cp -r outputs /content/drive/MyDrive/qwen2vl-cb-v2/ 2>/dev/null || true
!zip -r -q results-v2.zip out/qwen2vl-cb-v2 outputs
files.download('results-v2.zip')